In [ ]:
import torch

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch

import torch
from model import ResNet

net = ResNet(3, 11)
net.cpu()
# net.to(device)

# Nie możemy użyć kompilowanej sieci do eksportu do ONNX.
# Całe szczęście, możemy uzyskać oryginalny model przed kompilacją usuwając prefiksy z nazw warstw.
# net = torch.compile(net)
state_dict = torch.load('best.pth', map_location='cpu')

# Usuwanie prefiksów '_orig_mod.' lub innych technicznych nazw
new_state_dict = {}
for k, v in state_dict.items():
    name = k.replace('_orig_mod.', '') # usuwa prefiks kompilatora
    new_state_dict[name] = v

net.load_state_dict(new_state_dict)
# Konieczne jest ustawienie modelu w tryb ewaluacji przed eksportem
# żeby BatchNorm i Dropout działały poprawnie
net.eval()

/tmp/ipykernel_9190/3990960528.py:34: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 23 of general pattern rewrite rules.
Model checked successfully!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ManualTestPreprocess(nn.Module):
    def __init__(self):
        super().__init__()

        self.register_buffer(
            "mean", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1)
        )

    def forward(self, x):
        # x: (N, 3, H, W), float32, range [0,1]

        n, c, h, w = x.shape

        # --- Resize: shorter side -> 280 (aspect preserved)
        if h < w:
            new_h = 280
            new_w = int(w * 280 / h)
        else:
            new_w = 280
            new_h = int(h * 280 / w)

        x = F.interpolate(
            x,
            size=(new_h, new_w),
            mode="bilinear",
            align_corners=False
        )

        # --- CenterCrop (256, 256)
        top = (new_h - 256) // 2
        left = (new_w - 256) // 2
        x = x[:, :, top:top+256, left:left+256]

        # --- Normalize
        x = (x - self.mean) / self.std

        return x


net_with_transforms = nn.Sequential(ManualTestPreprocess(), net)
net_with_transforms.eval()

In [23]:

# 2. Correct Dummy Input (Batch=32, Channel=3, H=256, W=256)
dummy_input = torch.randn(32, 3, 256, 256)

# 3. Export with Dynamic Axes
torch.onnx.export(
    net_with_transforms,
    dummy_input,
    "model_with_transforms.onnx",
    export_params=True,        # store the trained parameter weights inside the model file
    do_constant_folding=True,  # whether to execute constant folding for optimization
    input_names=['input'],     # the model's input names
    output_names=['output'],   # the model's output names
    dynamic_axes={
        'input': {0: 'batch_size'},    # variable length axes
        'output': {0: 'batch_size'}
    }
)

import onnx

onnx_model = onnx.load("model_with_transforms.onnx")

onnx.checker.check_model(onnx_model)

print("Model checked successfully!")

/tmp/ipykernel_9190/2785995916.py:5: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 24 of general pattern rewrite rules.
Model checked successfully!


In [ ]:

import onnxruntime as ort
import numpy as np
import PIL.Image as Image
import torchvision

weather_images = torchvision.datasets.ImageFolder(root='dataset')

# 1. Tworzenie sesji (ładowanie modelu)
session = ort.InferenceSession("model.onnx")

# 2. Przygotowanie danych wejściowych
# 3. Pobranie nazw wejść i wyjść (zdefiniowałeś je przy eksporcie)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

from transforms import test_transform

test_transform_image = test_transform(
  Image.open('dataset/rainbow/0592.jpg').convert('RGB')
)

input_data = test_transform_image.unsqueeze(0).numpy().astype(np.float32)
# 4. Uruchomienie modelu
outputs = session.run([output_name], {input_name: input_data})

# 5. Interpretacja wyników
predictions = outputs[0]
predicted_class = np.argmax(predictions, axis=1)
print(f"Przewidziana klasa: {predicted_class}")

Przewidziana klasa: [7]


In [43]:
# Wyznaczanie jakości klasyfikacji dla poszczególnych klas
weather_images = torchvision.datasets.ImageFolder(root='dataset', transform=test_transform)
loader = torch.utils.data.DataLoader(weather_images, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
classes = weather_images.classes

# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}
total = 0
correct = 0
with torch.no_grad():
    for data in loader:
        images, labels = data # 'images' is a batch of 8 images
        
        # FIX 1: Use the actual batch images, not a static single image
        # Convert torch tensor to numpy for ONNX
        input_data = images.numpy().astype(np.float32)
        
        # 4. Run model on the batch
        outputs = session.run([output_name], {input_name: input_data})

        # 5. Interpret results
        logits = outputs[0]
        predictions = np.argmax(logits, axis=1) # Get the class indices

        # Collect the correct predictions for each class
        # Convert labels to numpy to iterate alongside predictions
        for label, pred in zip(labels.numpy(), predictions):
            if label == pred:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1
            
            # Global counters
            total += 1
            if label == pred:
                correct += 1
          
        print(f'Processed {total} images so far...')
        if total > 100:
            break
    # Use float division for accuracy to avoid 0% due to integer floor division
    overall_acc = 100 * correct / total
    print(f'Final accuracy: {overall_acc:.2f} %')

# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')
print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')


Processed 8 images so far...
Processed 16 images so far...
Processed 24 images so far...
Processed 32 images so far...
Processed 40 images so far...
Processed 48 images so far...
Processed 56 images so far...
Processed 64 images so far...
Processed 72 images so far...
Processed 80 images so far...
Processed 88 images so far...
Processed 96 images so far...
Processed 104 images so far...
Final accuracy: 86.54 %
Accuracy for class: dew   is 100.0 %
Accuracy for class: fogsmog is 90.9 %
Accuracy for class: frost is 62.5 %
Accuracy for class: glaze is 83.3 %
Accuracy for class: hail  is 90.0 %
Accuracy for class: lightning is 100.0 %
Accuracy for class: rain  is 88.9 %
Accuracy for class: rainbow is 100.0 %
Accuracy for class: rime  is 86.4 %
Accuracy for class: sandstorm is 90.0 %
Accuracy for class: snow  is 70.0 %
Accuracy of the network on the 10000 test images: 86 %
